# AI 第一课：从数据到第一个模型

**怎么跑：** 每个代码格子按 `Shift + Enter` 逐格运行，或者点格子左边的 ▶ 按钮。

**路线：** 环境自检 → numpy 数组运算 → pandas 表格 → matplotlib 看图找规律 → sklearn 训练第一个模型。

全程用的是 sklearn 自带的鸢尾花数据集（内置，不用联网下载）。

In [ ]:
# ---------- Part 0：环境自检 ----------
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

print("numpy      ", np.__version__)
print("pandas     ", pd.__version__)
print("matplotlib ", matplotlib.__version__)
print("scikit-learn", sklearn.__version__)

# 让 matplotlib 能显示中文（Windows 用 SimHei / 微软雅黑）
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

---
## Part 1：numpy —— 为什么 AI 不用列表，要用数组

AI 的本质就是**大量数字的批量运算**。Python 原生的 list 做不到「整体运算」，numpy 的 array 可以，而且底层是 C 实现的，快几十倍。

In [ ]:
# 先看 list 的"坑"
a = [1, 2, 3, 4]
print("list * 2 =", a * 2)      # 不是你想要的结果：它把列表重复了一遍

# numpy 数组才是真正的"数学向量"
arr = np.array([1, 2, 3, 4])
print("array * 2  =", arr * 2)
print("array + 10 =", arr + 10)
print("array 平方 =", arr ** 2)
print("求和/均值/最大:", arr.sum(), arr.mean(), arr.max())

In [ ]:
# 重点：下面这三行，就是神经网络里"一个神经元"的全部数学
x = np.array([1.0, 2.0, 3.0])      # 输入特征
w = np.array([0.5, 0.3, 0.2])      # 权重（模型要学的就是它）
b = 0.1                            # 偏置

y = np.dot(x, w) + b               # 加权求和
print("神经元输出:", y)

# 批量算：一次算 3 个样本（矩阵乘法）
X = np.array([[1.0, 2.0, 3.0],
              [0.5, 1.5, 2.5],
              [2.0, 1.0, 0.5]])
print("批量输出:", X @ w + b)      # @ 是矩阵乘法，等价于 np.dot(X, w)

> **记住这张图：** `输出 = 输入 × 权重 + 偏置`。后面所有模型（线性回归、神经网络、大模型）都是这个式子叠无数层、再加上一个「自动调整权重的算法」。

---
## Part 2：pandas —— 现实中的数据是表格

真实数据不会是一行数组，而是 Excel 那样的表格。pandas 的 `DataFrame` 就是 Python 里的 Excel。

In [ ]:
from sklearn.datasets import load_iris

# 加载鸢尾花数据集，as_frame=True 直接给 pandas 表格
iris = load_iris(as_frame=True)
df = iris.frame

print("表格形状（行, 列）:", df.shape)
df.head()      # 看前 5 行

In [ ]:
# 常用三板斧：info / describe / value_counts
print("--- 每列的统计摘要 ---")
print(df.describe())

print("\n--- 每个种类各有多少条 ---")
print(df['target'].value_counts())

print("\n--- 种类名称对照 ---")
for i, name in enumerate(iris.target_names):
    print(i, "=>", name)

In [ ]:
# 分组聚合：看三个种类各自的特征均值 —— 这一步已经在"找规律"了
df.groupby('target').mean().round(2)

In [ ]:
# ⚠️ pandas 3.0 重要提醒：不要用"链式赋值"改数据！
# 错误写法（在 pandas 3 里静默失效，不报错但数据没改）：
#     df['sepal length (cm)'][0] = 5.0

# 正确写法：用 .loc[行, 列]
df.loc[0, 'sepal length (cm)'] = 5.1
print("改完还是 5.1:", df.loc[0, 'sepal length (cm)'])

---
## Part 3：matplotlib —— 先「看见」规律，再谈模型

建模前一定要画图。**能用肉眼看出规律的，模型才学得会；你都看不出规律，模型大概率也学不到。**

In [ ]:
plt.figure(figsize=(8, 5))

for t, name in enumerate(iris.target_names):
    sub = df[df['target'] == t]
    plt.scatter(sub['sepal length (cm)'], sub['petal length (cm)'],
                label=name, alpha=0.7, s=40)

plt.xlabel('花萼长度 (cm)')
plt.ylabel('花瓣长度 (cm)')
plt.title('鸢尾花：三个种类的分布')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("👀 观察：setosa（蓝色）明显缩在左下角，和另两类分得很开——这就是模型能学会的规律。")

In [ ]:
# 再看每个特征的分布（直方图）
df.drop(columns=['target']).hist(figsize=(10, 6), bins=20, edgecolor='black')
plt.suptitle('四个特征的取值范围', y=1.02)
plt.tight_layout()
plt.show()

---
## Part 4：sklearn —— 你的第一个 AI 模型

任务：给一朵花测量 4 个数值，让它自动判断这是哪个种类。

这是**监督学习**：我们手里有「问题 + 标准答案」，让模型从里面学规律。

### 为什么要分训练集 / 测试集？

**模型不能拿考试题当作业做。** 拿 100% 的数据训练、再用同样的数据打分，得到的漂亮成绩是假的——它只是背下了答案。留出 20% 当「考试」，考试分才是真实水平。

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# X = 特征（题目），y = 标签（答案）
X = df.drop(columns=['target'])
y = df['target']

print("特征列名:", list(X.columns))

# 分训练集(80%)和测试集(20%)，random_state=42 保证每次分得一样，结果可复现
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"训练集 {len(X_train)} 条，测试集 {len(X_test)} 条")

In [ ]:
# KNN："近朱者赤"——看新样本周围最近的 3 个邻居属于哪类，就归为哪类
model = KNeighborsClassifier(n_neighbors=3)

model.fit(X_train, y_train)          # 训练：其实 KNN 只是把数据记住了
acc = model.score(X_test, y_test)    # 考试：在从没见过的数据上打分

print(f"测试集准确率: {acc:.2%}")

In [ ]:
# 用模型预测一朵新花
new_flower = pd.DataFrame(
    [[5.1, 3.5, 1.4, 0.2]],        # 花萼长/宽，花瓣长/宽
    columns=X.columns
)

pred = model.predict(new_flower)[0]
print("模型判断:", iris.target_names[pred])
proba = model.predict_proba(new_flower)[0]
print("把握(概率):", {str(name): round(float(p), 3) for name, p in zip(iris.target_names, proba)})

In [ ]:
# 看看模型在测试集上具体错在哪几条（混淆矩阵）
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

ConfusionMatrixDisplay(cm, display_labels=iris.target_names).plot(cmap='Blues')
plt.title('混淆矩阵：对角线越满越好')
plt.show()

---
## 🎯 动手改一改（这才是真正学会的部分）

别只是跑通，把下面每一项都改一次、跑一次、看结果怎么变：

1. **改 `n_neighbors`**：把 3 换成 1 和 50，准确率怎么变？想想为什么太大太小都不好。
2. **换模型**：把 `KNeighborsClassifier` 换成 `LogisticRegression(max_iter=200)`（记得 `from sklearn.linear_model import LogisticRegression`），对比一下。
3. **只用 2 个特征**：把 `X` 改成只保留花瓣长度和宽度两列（`X = df[['petal length (cm)', 'petal width (cm)']]`），准确率掉了吗？掉了多少？
4. **改 `random_state`**：换成 0、1、7，准确率会波动——这告诉你：单次准确率有偶然性。

### 💡 改的时候记住这个循环

```
改一个变量 → 跑 → 看数字怎么变 → 问自己为什么
```

AI 学习最重要的能力不是写代码，是**知道哪个旋钮会影响结果、往哪边拧**。